### Schema comparison function

In [0]:
from pyspark.sql.types import StructType

# Compare two schemas and return a list of the fields in one or the other but not both
def schema_difference(schema_a: StructType, schema_b: StructType, alias_a: str = "A", alias_b: str = "B"):
  ret = []
  a = schema_a.fieldNames()
  b = schema_b.fieldNames()
  for f in a:
      if f not in b:
         ret.append((alias_a, f)) 

  for f in b:
      if f not in a:
         ret.append((alias_b, f)) 
  return ret

In [0]:
# Set up widget to handle list of tables for processing, storage account, etc.
dbutils.widgets.text("TableList", "", "Table List")
dbutils.widgets.text("environment", "", "Environment")

## Setup paths to data files

In [0]:
%python
# =====================================================================================================================================
# Storage location
# ===================================================================================================================================
# Get environment and correct for prd / prod case
environment = dbutils.widgets.get('environment')
if environment.lower() == 'prd':
     environment = 'prod'
else: 
    environment = environment.lower()

# Set Azure blob file system secure protocol, storage account, and container
protocol = "abfss://"
store = f"sa{environment}bronzeingestion"

# Set catalog and schema to use; reset store if in sandbox
if environment == 'sbx':
    store = "staab09289802"
    spark.sql(f"use catalog hive_metastore;")
    spark.sql("use schema default;")
else:
    spark.sql(f"use catalog bronze_{environment};")
    spark.sql("use schema googleanalytics;")

container = "landing"

# Use the storage account access key instead of SAS token
spark.conf.set(f"fs.azure.account.key.{store}.dfs.core.windows.net", dbutils.secrets.get(scope="bronzeingestion-secret-scope", key=f"sa-{environment}-bronzeingestion-ak"))

# Build full path to files
rootpath = "GoogleBigQuery/GoogleAnalytics/fnd-cloud-project/analytics_250303278"
datapath = f"{protocol}{container}@{store}.dfs.core.windows.net/{rootpath}"

## Modify the way things are currently done
* Get schema of folder rather than a dataframe based on that
* Get current schema of destination table
* Compare and automate adding columns?

In [0]:
from collections import namedtuple

# Get the last loaded schema to use as the baseline for subsequent loading
last_load = spark.sql("select max(event_date) dt from ga_events").collect()
dt = last_load[0].dt
filepath = f"{datapath}/events/{dt[:-4]}/{dt[-4:-2]}/{dt[-2:]}/*"
sch_events = spark.read.format("parquet").load(filepath).schema
df_events = spark.createDataFrame([], sch_events)

# TODO: Figure out correct date to use for Users and PseudoUsers, replicate above logic for these
sch_users = spark.read.format("parquet").load(f"{datapath}/users/*/*/*").schema
df_users = spark.createDataFrame([], sch_users)

sch_pseudonymous_users = spark.read.format("parquet").load(f"{datapath}/pseudonymous_users/*/*/*").schema
df_pseudonymous_users = spark.createDataFrame([], sch_pseudonymous_users)

# Get distinct list of tables to process from notebook parameter / widget and sort them so processing is oldest to newest
tables = list(set(dbutils.widgets.get("TableList").split(",")).intersection(dbutils.widgets.get("TableList").split(",")))
tables.sort()

# Set catalog and schema to use
spark.sql("use catalog hive_metastore;")
spark.sql("use schema default;")

# Lists of each dataset by type - determine what to process based on schema differences
ev = []
us = []
pu = []
# A named tuple for capturing the schema summary of each dataset to consider processing
stat = namedtuple('stat', ['Dataset', 'NumFields', 'NumNewFields', 'filepath'])

# Loop through tables to process
for t in tables:
    # Break each table name into its folder structure, read the schema, and determine if schema has changed since last load
    #   ex: events_20240711 --> events/2024/07/11
    base = t.strip()[:-9]
    dt = t.strip()[-8:]
    filepath = f"{datapath}/{base}/{dt[:-4]}/{dt[-4:-2]}/{dt[-2:]}/*"

    try:
        # Reset the schema to test and the difference result
        sch_test = None
        diff = None

        # Read the schema of the table
        sch_test = spark.read.format("parquet").load(filepath).schema
        
        # Compare the schema to last known good one and append a results for subsequent decision
        if base == 'events':
            diff = schema_difference(sch_events, sch_test)
            ev.append(stat(t, len(sch_test), len(diff), filepath))

        # TODO: For User and PseudoUsers - Currently using first in list for comparison
        # need to prove which date to use for last load...
        elif base == 'users':
            if sch_users == None:
                sch_users = sch_test
                diff = None
            else:
                diff = schema_difference(sch_users, sch_test)
            # Record the status for Users
            us.append(stat(t, len(sch_test), len(diff), filepath))

        elif base == 'pseudonymous_users':
            if sch_pseudonymous_users == None:
                sch_pseudonymous_users = sch_test
                diff = None
            else:
                diff = schema_difference(sch_pseudonymous_users, sch_test)
            # Record the status for Pseudonymous Users
            pu.append(stat(t, len(sch_test), len(diff), filepath))

    except:
        print(f"File path does not exist, skipping {filepath}")

for e in ev:
    if e.NumNewFields == 0:
        df_events = df_events.union(spark.read.format("parquet").load(e.filepath))

for u in us:
    if u.NumNewFields == 0:
        df_users = df_users.union(spark.read.format("parquet").load(u.filepath))

for u in pu:
    if u.NumNewFields == 0:
        df_pseudonymous_users = df_pseudonymous_users.union(spark.read.format("parquet").load(u.filepath))

# Temporary views for subsequent data loading
df_events.createOrReplaceTempView("vwEvents")
df_users.createOrReplaceTempView("vwUsers")
df_pseudonymous_users.createOrReplaceTempView("vwPseudoUsers")

# Print summary of datasets to process or skip
print(f"""events:
{df_events.count()} rows
    {[ds.Dataset for ds in ev if ds.NumNewFields == 0]}
    Skipping: {[ds.Dataset for ds in ev if ds.NumNewFields > 0]}

users:
{df_users.count()} rows
    {[ds.Dataset for ds in us if ds.NumNewFields == 0]}
    Skipping: {[ds.Dataset for ds in us if ds.NumNewFields > 0]}

pseudonymous_users:
{df_pseudonymous_users.count()} rows
    {[ds.Dataset for ds in pu if ds.NumNewFields == 0]}
    Skipping: {[ds.Dataset for ds in pu if ds.NumNewFields > 0]}""")

In [0]:
# Get the schema for each dataset from table in hive metastore (TODO: switch to unity catalog when available)
msg = []
for d in dbutils.fs.ls(f"{datapath}"):
    if d.name == 'events/':
        msg.append(f"Found [{d.name}] source files.")
    elif d.name == 'users/':
        msg.append(f"Found [{d.name}] source files.")
    elif d.name == 'pseudonymous_users/':
        msg.append(f"Found [{d.name}] source files.")
    else:
        print(f"New dataset encountered!  Please add support for {d.name}")


## Figure out the schema differences between 10, 11, and 16 July 2024

In [0]:
df_a = None
df_b = None
path_a = "2024/07/16"
path_b = "2024/10/06"

df_a = spark.createDataFrame([], spark.read.format("parquet").load(f"{datapath}/events/{path_a}/*.parquet").schema)
df_b = spark.createDataFrame([], spark.read.format("parquet").load(f"{datapath}/events/{path_b}/*.parquet").schema)

exc = '''
print("===============================================")
i = 0
for b in df_b.schema.fields:
    i += 1
    if b in df_a.schema.fields:
        print(b)
#        print(f"{i} - Match: [{b.name}]")
        z = 0
    else:
        print(f"{i} - NEW:   [{b.name}]")
        df_a.schema.add(b)

print("===============================================")
j = 0
for a in df_a.schema.fields:
    j += 1
    print(f"{j} - {a}")
'''

comp = schema_difference(df_a.schema, df_b.schema, path_a, path_b)
if comp.__len__() > 0:
    print(f"{comp.__len__()} differences found\n===========================")
    for c in comp:
        print(c)
else:
    print("No differences found")

## Fiddle with data and schema creation, addition of columns...

In [0]:
df_a = spark.read.format("parquet").load(f"{datapath}/events/{path_a}/*.parquet")
df_b = spark.read.format("parquet").load(f"{datapath}/events/{path_b}/*.parquet")

df_a.createOrReplaceTempView("vwA")
df_b.createOrReplaceTempView("vwB")

In [0]:
%sql

select          cast(from_json(a.col.v.f[1].v, 'STRUCT<f: ARRAY<STRUCT<v: STRING>>>').f[1].v as int) ga_session_id
          ,     a.*
          ,     e.event_params, e.items, e.user_properties
from            vwA e
lateral view    posexplode_outer(from_json(event_params, 'STRUCT<v: ARRAY<STRUCT<v: STRUCT<f: ARRAY<STRUCT<v: STRING>>>>>>').v) a
where           a.col.v.f[0].v = 'ga_session_id'
limit 10

In [0]:
%sql

select          event_timestamp
          -- ,     cast(from_json(a.col.v.f[1].v, 'STRUCT<f: ARRAY<STRUCT<v: STRING>>>').f[1].v as int) ga_session_id
          -- ,     schema_of_json(e.event_params)
            ,       k.col key
            ,       _str.col string_value
            ,       cast(_int.col as int) int_value
            ,       cast(_dbl.col as double) double_value
            ,       cast(_flt.col as float) float_value
from            vwB e
lateral view    posexplode_outer(from_json(event_params, 'ARRAY<STRUCT<key: STRING, value: STRING>>')) a0
lateral view    posexplode_outer(from_json(a0.col.key, 'ARRAY<STRING>')) k
lateral view    posexplode_outer(from_json(a0.col.value, 'ARRAY<STRUCT<double_value: STRING, float_value: STRING, int_value: STRING, string_value: STRING>>')) a1
lateral view    posexplode_outer(from_json(a1.col.string_value, 'ARRAY<STRING>')) _str
lateral view    posexplode_outer(from_json(a1.col.int_value, 'ARRAY<STRING>')) _int
lateral view    posexplode_outer(from_json(a1.col.double_value, 'ARRAY<STRING>')) _dbl
lateral view    posexplode_outer(from_json(a1.col.float_value, 'ARRAY<STRING>')) _flt
where           k.pos = _str.pos
        and     k.pos = _int.pos
        and     k.pos = _flt.pos
        and     k.pos = _dbl.pos
--        and     k.col = 'ga_session_id'
        and     event_timestamp in (1728225655419784)

In [0]:
dbutils.notebook.exit('Skipping remaining cells')

import pyspark.sql.types
from pyspark.sql.types import *

ta = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(len(ta))

fnd = 12
if fnd in ta:
    print(f"Found {fnd}")

df1 = None
df2 = None
tmp = None

df1 = spark.createDataFrame(ta, ['id'])
# df1.printSchema()

sch = StructType([StructField('jingle', StringType(), False)])
sch.add(StructField('jangle', StringType(), False))

tmp = spark.createDataFrame([('Stupid', 'and Dumb')], sch)

sch2 = tmp.schema.add(StructField('jungle', StringType(), False))
df2 = spark.createDataFrame([('Stupid', 'and Dumb', 'None')], sch2)

# df2.printSchema()

x = schema_difference(df1.schema, sch2)
x